## PHASE 2 - DATA CLEANING WITH PANDAS

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
data=pd.read_csv("C:\\Users\\attil\\Downloads\\github_illusion_raw.csv")

In [19]:
data.shape

(105000, 21)

##### Dropping Duplicate rows

In [20]:
data.drop_duplicates(subset='username',keep='first',inplace=True)

In [21]:
data.shape

(101807, 21)

##### Fix: account_created

In [22]:
data['account_created']=pd.to_datetime(data['account_created'],format='mixed')

In [23]:
data['account_created'].head()

0   2021-08-17
1   2012-05-25
2   2022-07-21
3   2009-11-02
4   2015-11-23
Name: account_created, dtype: datetime64[ns]

In [24]:
data[data['account_created']>pd.Timestamp.now()]

,username,account_created,last_commit_date,total_repos,total_commits,commit_streak,avg_commit_hour,stars,forks,star_fork_ratio,...,following,readme_score,language,country,has_bio,has_profile_pic,bot_like_score,suspicion_score,profile_age_days,_profile_type


##### Fix: last_commit_date

In [ ]:
data['last_commit_date']=pd.to_datetime(data['last_commit_date'],format='mixed',errors='coerce')

##### Fix: commit_streak

In [ ]:
data['commit_streak']=data['commit_streak'].astype('str').str.replace(' days','')

In [ ]:
data['commit_streak']=data['commit_streak'].astype('float64')

In [ ]:
data.dtypes

##### Fix: language

In [ ]:
data['language']=data['language'].astype(str).str.lower().str.strip()
data['language']=data['language'].replace(['none','n/a','','nan'],'unknown')
data['language']=data['language'].str.title()
data['language']=data['language'].astype('category')

##### Fix: Country

In [ ]:
country_map = {
    'in': 'India', 'ind': 'India', 'india': 'India',
    'us': 'United States', 'usa': 'United States', 'united states': 'United States',
    'cn': 'China', 'china': 'China',
    'de': 'Germany', 'germany': 'Germany',
    'uk': 'United Kingdom', 'united kingdom': 'United Kingdom',
    'br': 'Brazil', 'brazil': 'Brazil',
    'ca': 'Canada', 'canada': 'Canada',
    'fr': 'France', 'france': 'France',
    'ru': 'Russia', 'russia': 'Russia',
    'jp': 'Japan', 'japan': 'Japan',
    'au': 'Australia', 'australia': 'Australia',
}

data['country']=data['country'].str.lower().str.strip()
data['country']=data['country'].map(country_map).fillna('Unknown')

In [ ]:
data['country']=data['country'].astype('category').str.title()

##### Fix: star_fork_ratio

In [ ]:
data['star_fork_ratio']=(data['stars']/data['forks']).replace([np.inf,-np.inf],np.nan)

##### Fix: avg_commit_hour

In [ ]:
data.loc[(data['avg_commit_hour']<0) | (data['avg_commit_hour']>23),'avg_commit_hour']=np.nan
median=data['avg_commit_hour'].median()
data['avg_commit_hour']=data['avg_commit_hour'].fillna(median)

##### Fix: bot_like_score

In [ ]:
data['bot_like_score'] = data['bot_like_score'].astype(str).str.lower().str.strip()
data['bot_like_score'] = data['bot_like_score'].replace({'1':'1','yes':'1','true':'1','0':'0','no':'0','false':'0','maybe':np.nan,'nan':np.nan})
data['bot_like_score'] = pd.to_numeric(data['bot_like_score'], errors='coerce').astype('Int64')

##### Fix: profile_age_days

In [ ]:
data.loc[data['profile_age_days']<1,'profile_age_days']=np.nan
data.loc[data['profile_age_days']>6000,'profile_age_days']=np.nan
data['profile_age_days']=data['profile_age_days'].fillna(data['profile_age_days'].median())

##### Fix: stars, forks, has_bio, has_profile_pic, 
#####      followers, following, readme_score, total_repos, total_commits, commit_streak


In [ ]:
for col in ['stars','forks','has_bio','has_profile_pic']:
    data[col]=data[col].fillna(0).astype(int)

for col in ['followers','following','readme_score','total_repos','total_commits','commit_streak']:
    data[col]=data[col].fillna(data[col].median())

### Feature Engineering

In [ ]:
data['account_age_years']=(pd.Timestamp.now()-data['account_created']).dt.days/365.25

In [ ]:
data['commit_frequency']=(data['total_commits']/data['account_age_years']).replace([np.inf,-np.inf],np.nan)
data['commit_frequency']=data['commit_frequency'].fillna(0)

In [ ]:
data['follow_ratio']=(data['followers']/data['following']).replace([np.inf,-np.inf],np.nan)
data['follow_ratio']=data['follow_ratio'].fillna(0)

In [ ]:
data['follow_ratio'].isin([np.inf, -np.inf]).sum()

In [ ]:
data['repo_commit_ratio']=(data['total_repos']/data['total_commits']).replace([np.inf,-np.inf],np.nan)
data['repo_commit_ratio']=data['repo_commit_ratio'].fillna(0)

In [ ]:
data['suspicious_hrs']=data['avg_commit_hour'].apply(lambda j: 1 if (j<=4 or j>=23) else 0)

##### Compute Suspicion score

In [ ]:
def compute_suspicion(row):
    score = 0

    if row['star_fork_ratio'] > 50:
        score += 0.25

    if row['follow_ratio'] < 0.05:
        score += 0.20

    if row['readme_score'] < 3:
        score += 0.20

    if row['suspicious_hrs'] == 1:
        score += 0.15

    if row['commit_streak'] in [100, 200, 300, 365, 500]:
        score += 0.10

    if row['has_bio'] == 0:
        score += 0.05

    if row['has_profile_pic'] == 0:
        score += 0.05

    return round(min(score, 1.0), 3)

mask = data['suspicion_score'].isnull()

data.loc[mask, 'suspicion_score'] = data[mask].apply(
    compute_suspicion,
    axis=1
)

In [ ]:
data['is_suspicious']=data['suspicion_score'].apply(lambda x: 1 if x>=0.5 else 0)

In [ ]:
data['is_suspicious'].value_counts()

### Export clean data

In [ ]:
data.to_csv("C:\\Users\\attil\\Downloads\\github_illusion_clean.csv")